## Tokenizing text

In [1]:
import os
import urllib.request

if not os.path.exists('the-verdict.txt'):
    url = ("https://raw.githubusercontent.com/rasbt/"
           "LLMs-from-scratch/main/ch02/01_main-chapter-code/"
           "the-verdict.txt")
    file_path = 'the-verdict.txt'
    urllib.request.urlretrieve(url, file_path)

In [2]:
with open('the-verdict.txt', 'r', encoding='utf-8') as f:
    raw_text = f.read()

print(f'Total number of character: {len(raw_text)}')

Total number of character: 20479


In [3]:
import re

text = "Hello, world. This, is a test."
result = re.split(r'(\s)', text)
print(result)

['Hello,', ' ', 'world.', ' ', 'This,', ' ', 'is', ' ', 'a', ' ', 'test.']


In [4]:
result = re.split(r'([,.]|\s)', text)
print(result)

['Hello', ',', '', ' ', 'world', '.', '', ' ', 'This', ',', '', ' ', 'is', ' ', 'a', ' ', 'test', '.', '']


In [5]:
result = [item for item in result if item.strip()]
print(result)

['Hello', ',', 'world', '.', 'This', ',', 'is', 'a', 'test', '.']


In [6]:
text = "Hello, world. Is this-- a test?"
result = re.split(r"([,:;?_!\"()\']|--|\s)", text)
result = [item for item in result if item.strip()]
print(result)

['Hello', ',', 'world.', 'Is', 'this', '--', 'a', 'test', '?']


In [7]:
preprocessed = re.split(r"([,:;?_!\"()\']|--|\s)", raw_text)
preprocessed = [item for item in preprocessed if item.strip()]
print(len(preprocessed))

4500


In [8]:
print(preprocessed[:30])

['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough', '--', 'so', 'it', 'was', 'no', 'great', 'surprise', 'to', 'me', 'to', 'hear', 'that', ',', 'in']


## Converting tokens into token IDs

In [9]:
all_words = sorted(set(preprocessed))
vocab_size = len(all_words)
print(vocab_size)

1212


In [10]:
vocab = {
    token:integer for integer, token in enumerate(all_words)
}

for key, value in enumerate(vocab.items()):
    print(value)
    if key >= 5:
        break

('!', 0)
('"', 1)
("'", 2)
('(', 3)
(')', 4)
(',', 5)


## Simple Tokenizer

In [11]:
class SimpleTokenizerV1:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i:s for s, i in vocab.items()}

    def encode(self, text):
        preprocessed = re.split(r"([,:;?_!\"()\']|--|\s)", text)
        preprocessed = [
            item.strip() for item in preprocessed if item.strip()
        ]
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids

    def decode(self, ids):
        text = ' '.join([self.int_to_str[i] for i in ids])
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
        return text

In [13]:
tokenizer = SimpleTokenizerV1(vocab)
text = """"It's the last he painted, you know," 
 Mrs. Gisburn said with pardonable pride."""
ids = tokenizer.encode(text)
print(ids)

[1, 57, 2, 907, 1058, 638, 559, 793, 5, 1208, 631, 5, 1, 68, 38, 908, 1188, 803, 847]


In [14]:
print(tokenizer.decode(ids))

" It' s the last he painted, you know," Mrs. Gisburn said with pardonable pride.


In [15]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokens = tokenizer.tokenize("I love Bangladesh!")
ids = tokenizer.encode("I love Bangladesh!")

print(tokens)  # ['I', 'Ġlove', 'ĠBang', 'ladesh', '!']
print(ids)     # [40, 2061, 13666, 24952, 0]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

['I', 'Ġlove', 'ĠBangladesh', '!']
[40, 1842, 19483, 0]


In [16]:
text = "Hello, do you like tea?"
print(tokenizer.encode(text))

KeyError: 'Hello'

## Adding special context tokens

In [16]:
all_tokens = sorted(list(set(preprocessed)))
all_tokens.extend(["<|endoftext|>", "<|unk|>"])
vocab = {
    token:integer for integer, token in enumerate(all_tokens)
}
print(len(vocab.items()))

1214


In [17]:
for i, item in enumerate(list(vocab.items())[-5:]):
    print(item)

('younger', 1209)
('your', 1210)
('yourself', 1211)
('<|endoftext|>', 1212)
('<|unk|>', 1213)


In [18]:
class SimpleTokenizerV2:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i:s for s, i in vocab.items()}

    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        preprocessed = [
            item.strip() for item in preprocessed if item.strip()
        ]
        preprocessed = [item if item in self.str_to_int else "<|unk|>"
                       for item in preprocessed]
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids

    def decode(self, ids):
        text = ' '.join([self.int_to_str[i] for i in ids])
        text = re.sub(r'\s+([,.:;?!"()\'])', r'\1', text)
        return text

In [19]:
text1 = "Hello, do you like tea?"
text2 = "In the sunlit terraces of the palace."
text = " <|endoftext|> ".join((text1, text2))
print(text)

Hello, do you like tea? <|endoftext|> In the sunlit terraces of the palace.


In [20]:
tokenizer = SimpleTokenizerV2(vocab)
print(tokenizer.encode(text))

[1213, 5, 373, 1208, 667, 1043, 10, 1212, 56, 1058, 1022, 1053, 766, 1058, 1213, 7]


In [21]:
print(tokenizer.decode(tokenizer.encode(text)))

<|unk|>, do you like tea? <|endoftext|> In the sunlit terraces of the <|unk|>.


## Byte pair encoding

In [22]:
import tiktoken

In [23]:
tokenizer = tiktoken.get_encoding('gpt2')

In [24]:
text = (
    "Hello, do you like tea? <|endoftext|> In the sunlit terraces"
     "of someunknownPlace."
)

integers = tokenizer.encode(text, allowed_special={"<|endoftext|>"})
print(integers)

[15496, 11, 466, 345, 588, 8887, 30, 220, 50256, 554, 262, 4252, 18250, 8812, 2114, 1659, 617, 34680, 27271, 13]


In [26]:
strings = tokenizer.decode(integers)

print(strings)

Hello, do you like tea? <|endoftext|> In the sunlit terracesof someunknownPlace.


## Data Sampling with sliding window

In [25]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()
enc_text = tokenizer.encode(raw_text)
print(len(enc_text))

5145


In [27]:
enc_sample = enc_text[50:]
# print(enc_sample)

In [28]:
context_size = 4
x = enc_sample[:context_size]
y = enc_sample[1:context_size+1]

print(f"x: {x}")
print(f"y: {y}")

x: [290, 4920, 2241, 287]
y: [4920, 2241, 287, 257]


In [29]:
for i in range(1, context_size+1):
    context = enc_sample[:i]
    desired = enc_sample[i]
    print(context, "---->", desired)

[290] ----> 4920
[290, 4920] ----> 2241
[290, 4920, 2241] ----> 287
[290, 4920, 2241, 287] ----> 257


In [30]:
for i in range(1, context_size+1):
    context = enc_sample[:i]
    desired = enc_sample[i]
    print(tokenizer.decode(context)," ----->", tokenizer.decode([desired]))

 and  ----->  established
 and established  ----->  himself
 and established himself  ----->  in
 and established himself in  ----->  a


In [31]:
import torch
from torch.utils.data import Dataset, DataLoader

In [35]:
class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        token_ids = tokenizer.encode(txt)

        # Use sliding window
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1: i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))
            
    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

In [36]:
def create_dataloader_v1(txt, batch_size=4, max_length=256,
                         stride=128, shuffle=True, drop_last=True,
                         num_workers=0):
    tokenizer = tiktoken.get_encoding('gpt2')
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)
    dataloader = DataLoader(
         dataset,
         batch_size=batch_size,
         shuffle=shuffle,
         drop_last=drop_last, 
         num_workers=num_workers 
         )
    return dataloader

In [37]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

In [40]:
dataloader = create_dataloader_v1(
    raw_text, batch_size=3, max_length=3, stride=1, shuffle=False)

data_iter = iter(dataloader)
print(next(data_iter))

[tensor([[  40,  367, 2885],
        [ 367, 2885, 1464],
        [2885, 1464, 1807]]), tensor([[ 367, 2885, 1464],
        [2885, 1464, 1807],
        [1464, 1807, 3619]])]


In [41]:
second_batch = next(data_iter)
print(second_batch)

[tensor([[1464, 1807, 3619],
        [1807, 3619,  402],
        [3619,  402,  271]]), tensor([[ 1807,  3619,   402],
        [ 3619,   402,   271],
        [  402,   271, 10899]])]


In [42]:
dataloader = create_dataloader_v1(
 raw_text, batch_size=8, max_length=4, stride=4,
 shuffle=False
)
data_iter = iter(dataloader)
inputs, targets = next(data_iter)
print("Inputs:\n", inputs)
print("\nTargets:\n", targets)

Inputs:
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])

Targets:
 tensor([[  367,  2885,  1464,  1807],
        [ 3619,   402,   271, 10899],
        [ 2138,   257,  7026, 15632],
        [  438,  2016,   257,   922],
        [ 5891,  1576,   438,   568],
        [  340,   373,   645,  1049],
        [ 5975,   284,   502,   284],
        [ 3285,   326,    11,   287]])


## Creating Token embedding

In [43]:
# demo embedding example
input_ids = torch.tensor([[2, 3, 5, 1]])
vocab_size = 6
output_dim = 3

torch.manual_seed(123)
embedding_layer = torch.nn.Embedding(vocab_size, output_dim)
print(embedding_layer.weight)

Parameter containing:
tensor([[ 0.3374, -0.1778, -0.1690],
        [ 0.9178,  1.5810,  1.3010],
        [ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-1.1589,  0.3255, -0.6315],
        [-2.8400, -0.7849, -1.4096]], requires_grad=True)


In [46]:
print(embedding_layer(input_ids))

tensor([[[ 1.2753, -0.2010, -0.1606],
         [-0.4015,  0.9666, -1.1481],
         [-2.8400, -0.7849, -1.4096],
         [ 0.9178,  1.5810,  1.3010]]], grad_fn=<EmbeddingBackward0>)


## Encoding word positions

In [44]:
vocab_size = 50257
output_dim = 256
token_embedding_layer = torch.nn.Embedding(vocab_size, output_dim)

In [45]:
max_length = 4
dataloader = create_dataloader_v1(
 raw_text, batch_size=8, max_length=max_length,
 stride=max_length, shuffle=False
)
data_iter = iter(dataloader)
inputs, targets = next(data_iter)
print("Token IDs:\n", inputs)
print("\nInputs shape:\n", inputs.shape)

Token IDs:
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])

Inputs shape:
 torch.Size([8, 4])


In [61]:
token_embeddings = token_embedding_layer(inputs)
print(token_embeddings.shape) #(batch size, context length, output dim)

torch.Size([8, 4, 256])


In [62]:
context_length = max_length
pos_embedding_layer = torch.nn.Embedding(context_length, output_dim)
pos_embeddings = pos_embedding_layer(torch.arange(context_length))
print(pos_embeddings.shape)

torch.Size([4, 256])


In [66]:
input_embeddings = token_embeddings + pos_embeddings
print(input_embeddings.shape)

torch.Size([8, 4, 256])


In [47]:
class SimpleTokenizerV3:
    def __init__(self, vocab: dict, unk_token="<|unk|>", bos_token=None, eos_token=None):
        self.str_to_int = vocab
        self.int_to_str = {i:s for s, i in vocab.items()}
        self.unk_token = unk_token
        self.bos_token = bos_token
        self.eos_token = eos_token

        # Ensure special tokens exist in vocab
        for tok in [unk_token, bos_token, eos_token]:
            if tok and tok not in vocab:
                idx = len(self.str_to_int)
                self.str_to_int[tok] = idx
                self.int_to_str[idx] = tok

    def _basic_tokenize(self, text):
        tokens = re.split(r"([,.:;?_!\"()\']|--|\s)", text)
        return [tok.strip() for tok in tokens if tok.strip()]

    def encode(self, text):
        tokens = self._basic_tokenize(text)
        tokens = [
            tok if tok in self.str_to_int else self.unk_token
            for tok in tokens
        ]

        if self.bos_token:
            tokens.insert(0, self.bos_token)
        if self.eos_token:
            tokens.append(self.eos_token)
        return [self.str_to_int[tok] for tok in tokens]

    def decode(self, ids):
        tokens = [self.int_to_str.get(i, self.unk_token) for i in ids]
        text = ' '.join(tokens)
        return re.sub(r'\s+([,.?!"()\'])', r'\1', text)

In [48]:
class GPTDataset(Dataset):
    def __init__(self, text, tokenizer, max_length=256, stride=128):
        self.input_ids = []
        self.target_ids = []

        token_ids = tokenizer.encode(text)

        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1: i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))
            
    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

In [49]:
def create_dataloader(
    text, tokenizer, batch_size=4, max_length=256,
    stride=128, shuffle=True, drop_last=True, num_workers=0
):
    dataset = GPTDataset(text, tokenizer, max_length, stride)
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers
    )

In [50]:
tokenizer = SimpleTokenizerV3(vocab, unk_token="<|unk|>", bos_token="<|bos|>", eos_token="<|eos|>")

In [53]:
# Encode
text = "In the sunlit terraces of the palace."
ids = tokenizer.encode(text)
print(ids)  # → includes <|unk|> for "Dhaka"

[1214, 56, 1058, 1022, 1053, 766, 1058, 1213, 7, 1215]


In [54]:
# Decode
print(tokenizer.decode(ids))

<|bos|> In the sunlit terraces of the <|unk|>. <|eos|>
